# Feature Selection -- Filter Based
Feature Selection is the process of reducing the number of features when developing a predictive model.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import kagglehub
path = kagglehub.dataset_download("uciml/human-activity-recognition-with-smartphones")

Using Colab cache for faster access to the 'human-activity-recognition-with-smartphones' dataset.


In [ ]:
df = pd.read_csv(path + "/train.csv").drop("subject", axis=1)

In [ ]:
df

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-skewness(),fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",Activity
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.298676,-0.710304,-0.112754,0.030400,-0.464761,-0.018446,-0.841247,0.179941,-0.058627,STANDING
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,-0.595051,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317,STANDING
2,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,-0.963668,-0.977469,-0.938692,...,-0.390748,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118,STANDING
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.982750,-0.989302,-0.938692,...,-0.117290,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663,STANDING
4,0.276629,-0.016570,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,...,-0.351471,-0.699205,0.123320,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892,STANDING
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7347,0.299665,-0.057193,-0.181233,-0.195387,0.039905,0.077078,-0.282301,0.043616,0.060410,0.210795,...,-0.588433,-0.880324,-0.190437,0.829718,0.206972,-0.425619,-0.791883,0.238604,0.049819,WALKING_UPSTAIRS
7348,0.273853,-0.007749,-0.147468,-0.235309,0.004816,0.059280,-0.322552,-0.029456,0.080585,0.117440,...,-0.390738,-0.680744,0.064907,0.875679,-0.879033,0.400219,-0.771840,0.252676,0.050053,WALKING_UPSTAIRS
7349,0.273387,-0.017011,-0.045022,-0.218218,-0.103822,0.274533,-0.304515,-0.098913,0.332584,0.043999,...,0.025145,-0.304029,0.052806,-0.266724,0.864404,0.701169,-0.779133,0.249145,0.040811,WALKING_UPSTAIRS
7350,0.289654,-0.018843,-0.158281,-0.219139,-0.111412,0.268893,-0.310487,-0.068200,0.319473,0.101702,...,0.063907,-0.344314,-0.101360,0.700740,0.936674,-0.589479,-0.785181,0.246432,0.025339,WALKING_UPSTAIRS


In [ ]:
df['Activity'].value_counts()

,count
Activity,
LAYING,1407
STANDING,1374
SITTING,1286
WALKING,1226
WALKING_UPSTAIRS,1073
WALKING_DOWNSTAIRS,986


## Apply Logistic Regression

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Separate features and target
X = df.drop('Activity', axis=1)
y = df['Activity']

# Encode target labels
le = LabelEncoder()
y = le.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape, X_test.shape

((5881, 561), (1471, 561))

In [ ]:
# Initialize and train logistic regression model
log_reg = LogisticRegression(max_iter=1000)  # Increase max_iter if it doesn't converge
log_reg.fit(X_train, y_train)

# Make predictions on the test set
y_pred = log_reg.predict(X_test)

# Calculate and print accuracy score
accuracy = accuracy_score(y_test, y_pred)
print("Test accuracy:", accuracy)

Test accuracy: 0.9802855200543847


## Duplicate Features

In [ ]:
def get_duplicate_columns(df):

    duplicate_columns = {}
    seen_columns = {}

    for column in df.columns:
        current_column = df[column]

        # Convert column data to bytes
        try:
            current_column_hash = current_column.values.tobytes()
        except AttributeError:
            current_column_hash = current_column.to_string().encode()

        if current_column_hash in seen_columns:
            if seen_columns[current_column_hash] in duplicate_columns:
                duplicate_columns[seen_columns[current_column_hash]].append(column)
            else:
                duplicate_columns[seen_columns[current_column_hash]] = [column]
        else:
            seen_columns[current_column_hash] = column

    return duplicate_columns

In [ ]:
duplicate_columns = get_duplicate_columns(X)

In [ ]:
duplicate_columns

{'tBodyAccMag-mean()': ['tBodyAccMag-sma()',
  'tGravityAccMag-mean()',
  'tGravityAccMag-sma()'],
 'tBodyAccMag-std()': ['tGravityAccMag-std()'],
 'tBodyAccMag-mad()': ['tGravityAccMag-mad()'],
 'tBodyAccMag-max()': ['tGravityAccMag-max()'],
 'tBodyAccMag-min()': ['tGravityAccMag-min()'],
 'tBodyAccMag-energy()': ['tGravityAccMag-energy()'],
 'tBodyAccMag-iqr()': ['tGravityAccMag-iqr()'],
 'tBodyAccMag-entropy()': ['tGravityAccMag-entropy()'],
 'tBodyAccMag-arCoeff()1': ['tGravityAccMag-arCoeff()1'],
 'tBodyAccMag-arCoeff()2': ['tGravityAccMag-arCoeff()2'],
 'tBodyAccMag-arCoeff()3': ['tGravityAccMag-arCoeff()3'],
 'tBodyAccMag-arCoeff()4': ['tGravityAccMag-arCoeff()4'],
 'tBodyAccJerkMag-mean()': ['tBodyAccJerkMag-sma()'],
 'tBodyGyroMag-mean()': ['tBodyGyroMag-sma()'],
 'tBodyGyroJerkMag-mean()': ['tBodyGyroJerkMag-sma()'],
 'fBodyAccMag-mean()': ['fBodyAccMag-sma()'],
 'fBodyBodyAccJerkMag-mean()': ['fBodyBodyAccJerkMag-sma()'],
 'fBodyBodyGyroMag-mean()': ['fBodyBodyGyroMag-sma()'

In [ ]:
for one_list in duplicate_columns.values():
    X_train.drop(columns=one_list,inplace=True)
    X_test.drop(columns=one_list,inplace=True)

In [ ]:
X_train.shape, X_test.shape

((5881, 540), (1471, 540))

## Variance Threshold

In [ ]:
from sklearn.feature_selection import VarianceThreshold
sel = VarianceThreshold(threshold=0.05)
sel.fit(X_train)

VarianceThreshold(threshold=0.05)

In [ ]:
sum(sel.get_support())

np.int64(349)

In [ ]:
columns = X_train.columns[sel.get_support()]
columns

Index(['tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z',
       'tBodyAcc-mad()-X', 'tBodyAcc-mad()-Y', 'tBodyAcc-mad()-Z',
       'tBodyAcc-max()-X', 'tBodyAcc-max()-Y', 'tBodyAcc-max()-Z',
       'tBodyAcc-min()-X',
       ...
       'fBodyBodyGyroJerkMag-meanFreq()', 'fBodyBodyGyroJerkMag-skewness()',
       'fBodyBodyGyroJerkMag-kurtosis()', 'angle(tBodyAccMean,gravity)',
       'angle(tBodyAccJerkMean),gravityMean)',
       'angle(tBodyGyroMean,gravityMean)',
       'angle(tBodyGyroJerkMean,gravityMean)', 'angle(X,gravityMean)',
       'angle(Y,gravityMean)', 'angle(Z,gravityMean)'],
      dtype='object', length=349)

In [ ]:
X_train = sel.transform(X_train)
X_test = sel.transform(X_test)

X_train = pd.DataFrame(X_train, columns=columns)
X_test = pd.DataFrame(X_test, columns=columns)

In [ ]:
X_train.shape, X_test.shape

((5881, 349), (1471, 349))

In [ ]:
# Initialize and train logistic regression model
log_reg = LogisticRegression(max_iter=1000)  # Increase max_iter if it doesn't converge
log_reg.fit(X_train, y_train)

# Make predictions on the test set
y_pred = log_reg.predict(X_test)

# Calculate and print accuracy score
accuracy = accuracy_score(y_test, y_pred)
print("Test accuracy:", accuracy)

Test accuracy: 0.981645139360979


## Correlation Coefficient

In [ ]:
def get_correlated_columns(df, threshold):
    corr_matrix = df.corr().abs()
    # Select upper triangle of correlation matrix
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Find features with correlation greater than threshold
    to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]
    return to_drop

correlated_columns = get_correlated_columns(X_train, 0.9)
print(f"Number of correlated columns to drop: {len(correlated_columns)}")

X_train.drop(columns=correlated_columns, inplace=True)
X_test.drop(columns=correlated_columns, inplace=True)

print(f"X_train shape after dropping correlated columns: {X_train.shape}")
print(f"X_test shape after dropping correlated columns: {X_test.shape}")

Number of correlated columns to drop: 237
X_train shape after dropping correlated columns: (5881, 112)
X_test shape after dropping correlated columns: (1471, 112)


In [ ]:
# Initialize and train logistic regression model again
log_reg = LogisticRegression(max_iter=1000) # Increase max_iter if it doesn't converge
log_reg.fit(X_train, y_train)

# Make predictions on the test set
y_pred = log_reg.predict(X_test)

# Calculate and print accuracy score
accuracy = accuracy_score(y_test, y_pred)
print("Test accuracy after correlation feature selection:", accuracy)

Test accuracy after correlation feature selection: 0.9680489462950373


## ANOVA

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

# Apply ANOVA F-value feature selection
# Selecting top 100 features for demonstration. The number of features can be tuned.
selector = SelectKBest(f_classif, k=100)

X_train_anova = selector.fit_transform(X_train, y_train)
X_test_anova = selector.transform(X_test)

# Get the names of the selected features
selected_features_anova = X_train.columns[selector.get_support()]

X_train = pd.DataFrame(X_train_anova, columns=selected_features_anova)
X_test = pd.DataFrame(X_test_anova, columns=selected_features_anova)

print(f"X_train shape after ANOVA feature selection: {X_train.shape}")
print(f"X_test shape after ANOVA feature selection: {X_test.shape}")

X_train shape after ANOVA feature selection: (5881, 100)
X_test shape after ANOVA feature selection: (1471, 100)


In [ ]:
# Initialize and train logistic regression model again
log_reg = LogisticRegression(max_iter=1000) # Increase max_iter if it doesn't converge
log_reg.fit(X_train, y_train)

# Make predictions on the test set
y_pred = log_reg.predict(X_test)

# Calculate and print accuracy score
accuracy = accuracy_score(y_test, y_pred)
print("Test accuracy after ANOVA feature selection:", accuracy)

Test accuracy after ANOVA feature selection: 0.9666893269884432


# Feature Selection -- Wrapper Based

## Exhaustive Feature Selection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('https://gist.githubusercontent.com/curran/a08a1080b88344b0c8a7/raw/0e7a9b0a5d22642a06d3d5b9bcbad9890c8ee534/iris.csv')
df

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop('species', axis=1)
y = df['species']

# Encode target labels
le = LabelEncoder()
y = le.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (120, 4)
X_test shape: (30, 4)
y_train shape: (120,)
y_test shape: (30,)


In [ ]:
from mlxtend.feature_selection import ExhaustiveFeatureSelector
from sklearn.linear_model import LogisticRegression

# Instantiate a Logistic Regression model
log_reg = LogisticRegression(solver='liblinear', random_state=42)

# Initialize ExhaustiveFeatureSelector
efs = ExhaustiveFeatureSelector(log_reg,
                               max_features=X_train.shape[1],
                               scoring='accuracy',
                               cv=5)

# Fit the selector to the training data
efs.fit(X_train, y_train)


Features: 15/15

ExhaustiveFeatureSelector(estimator=LogisticRegression(random_state=42,
                                                       solver='liblinear'),
                          feature_groups=[[0], [1], [2], [3]], max_features=4)

In [ ]:
from sklearn.metrics import accuracy_score

print('Best score:', efs.best_score_)
print('Best features (indices):', efs.best_idx_)
print('Best features (names):', X_train.columns[list(efs.best_idx_)].tolist())

# Transform the training and test data to include only the selected features
X_train_selected = efs.transform(X_train)
X_test_selected = efs.transform(X_test)

# Convert back to DataFrame with column names for clarity
X_train_selected = pd.DataFrame(X_train_selected, columns=X_train.columns[list(efs.best_idx_)])
X_test_selected = pd.DataFrame(X_test_selected, columns=X_test.columns[list(efs.best_idx_)])

# Initialize and train logistic regression model with selected features
log_reg_efs = LogisticRegression(solver='liblinear', random_state=42)
log_reg_efs.fit(X_train_selected, y_train)

# Make predictions on the test set
y_pred_efs = log_reg_efs.predict(X_test_selected)

# Calculate and print accuracy score
accuracy_efs = accuracy_score(y_test, y_pred_efs)
print("Test accuracy after Exhaustive Feature Selection:", accuracy_efs)


Best score: 0.9666666666666668
Best features (indices): (1, 2, 3)
Best features (names): ['sepal_width', 'petal_length', 'petal_width']
Test accuracy after Exhaustive Feature Selection: 0.9666666666666667


## Sequential Backward Elimination

In [ ]:
# load the dataset
data = pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv')

# separate the target variable
X = data.drop("medv", axis=1)
y = data['medv']

# split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

print(X_train.shape)

(404, 13)


In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
model = LinearRegression()
print("training",np.mean(cross_val_score(model, X_train, y_train, cv=5, scoring='r2')))
print("testing",np.mean(cross_val_score(model, X_test, y_test, cv=5, scoring='r2')))

training 0.7025123301096212
testing 0.6514899901155402


In [ ]:
from mlxtend.feature_selection import SequentialFeatureSelector

# Instantiate a LinearRegression model (already done in previous steps, reusing 'model' variable)
# model = LinearRegression()

# Initialize SequentialBackwardSelection (using SequentialFeatureSelector with backward=True)
sbs = SequentialFeatureSelector(model,
                                  k_features='best',
                                  forward=False, # For backward elimination
                                  floating=False,
                                  scoring='r2',
                                  cv=5,
                                  n_jobs=-1)

# Fit the selector to the training data
sbs.fit(X_train, y_train)

SequentialFeatureSelector(estimator=LinearRegression(), forward=False,
                          k_features=(1, 13), n_jobs=-1, scoring='r2')

In [ ]:
print('Best features (indices):', sbs.k_feature_idx_)

# Transform the training and test data to include only the selected features
X_train_selected_sbs = sbs.transform(X_train)
X_test_selected_sbs = sbs.transform(X_test)

# Initialize and train Linear Regression model with selected features
model_sbs = LinearRegression()
model_sbs.fit(X_train_selected_sbs, y_train)

# Make predictions on the test set
y_pred_sbs = model_sbs.predict(X_test_selected_sbs)

# Calculate and print R2 score
from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_pred_sbs)
print("R2 score after Sequential Backward Elimination:", r2)

Best features (indices): (0, 1, 4, 5, 7, 8, 9, 10, 11, 12)
R2 score after Sequential Backward Elimination: 0.7551933887467764


In [ ]:
print("training",np.mean(cross_val_score(model_sbs, X_train_selected_sbs, y_train, cv=5, scoring='r2')))
print("testing",np.mean(cross_val_score(model_sbs, X_test_selected_sbs, y_test, cv=5, scoring='r2')))

training 0.7100327839218561
testing 0.7205819296124483


## Sequantial Forward Selector

In [ ]:
from mlxtend.feature_selection import SequentialFeatureSelector

# Instantiate a LinearRegression model (already done in previous steps, reusing 'model' variable)
# model = LinearRegression()

# Initialize SequentialBackwardSelection (using SequentialFeatureSelector with backward=True)
sbs = SequentialFeatureSelector(model,
                                  k_features='best',
                                  forward=True, # For Forward Selector
                                  floating=False,
                                  scoring='r2',
                                  cv=5,
                                  n_jobs=-1)

# Fit the selector to the training data
sbs.fit(X_train, y_train)

SequentialFeatureSelector(estimator=LinearRegression(), k_features=(1, 13),
                          n_jobs=-1, scoring='r2')

In [ ]:
print('Best features (indices):', sbs.k_feature_idx_)

# Transform the training and test data to include only the selected features
X_train_selected_sbs = sbs.transform(X_train)
X_test_selected_sbs = sbs.transform(X_test)

# Initialize and train Linear Regression model with selected features
model_sbs = LinearRegression()
model_sbs.fit(X_train_selected_sbs, y_train)

# Make predictions on the test set
y_pred_sbs = model_sbs.predict(X_test_selected_sbs)

# Calculate and print R2 score
from sklearn.metrics import r2_score
r2 = r2_score(y_test, y_pred_sbs)
print("R2 score after Sequential Backward Elimination:", r2)

Best features (indices): (0, 1, 4, 5, 7, 8, 9, 10, 11, 12)
R2 score after Sequential Backward Elimination: 0.7551933887467764


In [ ]:
print("training",np.mean(cross_val_score(model_sbs, X_train_selected_sbs, y_train, cv=5, scoring='r2')))
print("testing",np.mean(cross_val_score(model_sbs, X_test_selected_sbs, y_test, cv=5, scoring='r2')))

training 0.7100327839218561
testing 0.7205819296124483


## Recursive Feature Elimination

In [ ]:
df = pd.read_csv('https://gist.githubusercontent.com/curran/a08a1080b88344b0c8a7/raw/0e7a9b0a5d22642a06d3d5b9bcbad9890c8ee534/iris.csv')
df

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [ ]:
X = df.drop('species', axis=1)
y = df['species']

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Encode target labels
le = LabelEncoder()
y = le.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (120, 4)
X_test shape: (30, 4)
y_train shape: (120,)
y_test shape: (30,)


In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

# 1. Instantiate a LogisticRegression model
log_reg_rfe = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)

# 2. Initialize the RFE selector
# For Iris dataset (4 features), selecting 2 features is a good demonstration.
rfe_selector = RFE(estimator=log_reg_rfe, n_features_to_select=2, step=1)

# 3. Fit the RFE selector to the training data
rfe_selector.fit(X_train, y_train)

# 4. Transform both the X_train and X_test datasets
X_train_rfe = rfe_selector.transform(X_train)
X_test_rfe = rfe_selector.transform(X_test)

# Get the names of the selected features
selected_features_rfe = X_train.columns[rfe_selector.support_]

# Convert the transformed arrays back to DataFrames with column names
X_train_rfe = pd.DataFrame(X_train_rfe, columns=selected_features_rfe)
X_test_rfe = pd.DataFrame(X_test_rfe, columns=selected_features_rfe)

print(f"Selected features by RFE: {list(selected_features_rfe)}")
print(f"X_train_rfe shape: {X_train_rfe.shape}")
print(f"X_test_rfe shape: {X_test_rfe.shape}")

Selected features by RFE: ['sepal_width', 'petal_width']
X_train_rfe shape: (120, 2)
X_test_rfe shape: (30, 2)


In [ ]:
from sklearn.metrics import accuracy_score

# Initialize and train logistic regression model with RFE selected features
log_reg_rfe_eval = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
log_reg_rfe_eval.fit(X_train_rfe, y_train)

# Make predictions on the test set with RFE selected features
y_pred_rfe = log_reg_rfe_eval.predict(X_test_rfe)

# Calculate and print accuracy score
accuracy_rfe = accuracy_score(y_test, y_pred_rfe)
print("Test accuracy after RFE feature selection:", accuracy_rfe)

Test accuracy after RFE feature selection: 0.9333333333333333


# Feature Selection -- Embedded Method

## LASSO

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv')
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LassoCV
from sklearn.metrics import accuracy_score

# Separate features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames to retain column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Apply LassoCV for feature selection
# LassoCV finds the optimal alpha (regularization strength) using cross-validation
lasso_selector = LassoCV(cv=5, random_state=42, max_iter=10000).fit(X_train_scaled, y_train)

# Get coefficients and identify selected features (non-zero coefficients)
selected_features_mask = lasso_selector.coef_ != 0
selected_feature_names = X_train.columns[selected_features_mask].tolist()

print(f"Optimal alpha: {lasso_selector.alpha_}")
print(f"Number of features selected by Lasso: {len(selected_feature_names)}")
print(f"Selected features: {selected_feature_names}")

# Filter X_train and X_test to include only selected features
X_train_lasso = X_train_scaled[selected_feature_names]
X_test_lasso = X_test_scaled[selected_feature_names]

# Train a Logistic Regression model with the selected features
log_reg_lasso = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
log_reg_lasso.fit(X_train_lasso, y_train)

# Make predictions on the test set
y_pred_lasso = log_reg_lasso.predict(X_test_lasso)

# Calculate and print accuracy score
accuracy_lasso = accuracy_score(y_test, y_pred_lasso)
print(f"Test accuracy after LASSO feature selection: {accuracy_lasso}")

Optimal alpha: 0.0074745106413323555
Number of features selected by Lasso: 7
Selected features: ['Pregnancies', 'Glucose', 'BloodPressure', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
Test accuracy after LASSO feature selection: 0.7402597402597403


## Decision Tree

In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectFromModel
import warnings
warnings.filterwarnings('ignore')

# Assuming df is already loaded from the previous LASSO section (diabetes.csv)
# If not, uncomment the following line:
# df = pd.read_csv('https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv')

# Separate features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale the features (important for many models, though not strictly for tree-based feature importance)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert scaled arrays back to DataFrames to retain column names
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Initialize a Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)

# Train the Decision Tree model on the scaled training data
dt_model.fit(X_train_scaled, y_train)

# Use SelectFromModel to select features based on feature importances
# You can adjust the 'threshold' parameter. 'median' or 'mean' are common choices.
# Here, we'll use a default of 'median'.
selector_dt = SelectFromModel(dt_model, prefit=True, threshold='mean')

# Transform the training and test data to include only the selected features
X_train_dt = selector_dt.transform(X_train_scaled)
X_test_dt = selector_dt.transform(X_test_scaled)

# Get the names of the selected features
selected_features_mask_dt = selector_dt.get_support()
selected_feature_names_dt = X_train.columns[selected_features_mask_dt].tolist()

print(f"Number of features selected by Decision Tree: {len(selected_feature_names_dt)}")
print(f"Selected features: {selected_feature_names_dt}")

# Train a Logistic Regression model with the selected features
log_reg_dt = LogisticRegression(solver='liblinear', random_state=42, max_iter=1000)
log_reg_dt.fit(X_train_dt, y_train)

# Make predictions on the test set
y_pred_dt = log_reg_dt.predict(X_test_dt)

# Calculate and print accuracy score
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print(f"Test accuracy after Decision Tree feature selection: {accuracy_dt}")

Number of features selected by Decision Tree: 3
Selected features: ['Glucose', 'BMI', 'Age']
Test accuracy after Decision Tree feature selection: 0.7272727272727273
